In [51]:
import pandas as pd
import json
import datetime as dt 


### Reading JSON files 

In [53]:
def read_bronze_data(resourse):
    path = f"../BRONZE/{resourse}.json"
    try:
        with open(path , 'r') as file:
            bronze_data = json.load(file)
            print(f'{resourse}.json successfully read!') 
            return pd.DataFrame(bronze_data['source_data'])
    except FileNotFoundError:
        print(f'{resourse}.json not found ')

In [54]:
launches_df = read_bronze_data('launches')
launchpads_df = read_bronze_data('launchpads')
rockets_df = read_bronze_data('rockets')

launches.json successfully read!
launchpads.json successfully read!
rockets.json successfully read!


### Fuctions for Tranformation


In [55]:
def get_launch_status(row):
    if row['upcoming'] == True:
        return 'Upcoming'
    elif row['success'] == True:
        return 'Success'
    elif row['success'] == False:
        return 'Failed'
    else:
        return 'Unknown'
    
def check_duplicates(df , col):
    dupes = df.duplicated(subset=[col]).sum()
    print(f"Duplicates in {col} : {dupes}")

def remove_duplicates(df , col):
    df.drop_duplicates(subset=[col], inplace=True)
    print(f"successfully removed duplicates in {col}")
    
def check_all_nulls(df):
    print(f"nulls : {df.isna().sum()}")

def check_nulls(df , col ):
     print(f"nulls : {df[col].isna().sum()}")

def referenence_check(df1 , df2 , col1 , col2):
    count = 0
    for i in df1[col1]:
        if i not in df2[col2].values:
            count += 1
    print(f"Reference check for {col1} in {col2} : {count} unmatched values")  

def range_check(df , col , min_val , max_val):
    out_of_range = df[(df[col] < min_val) | (df[col] > max_val)]
    print(f"Range check for {col} in range [{min_val}, {max_val}]: {len(out_of_range)} out of range values")

def success_col_check(df ,col):
    valid_values = df[(df[col] == True) | (df[col] == False) | (df[col].isna())]
    print(f"Success column check for {col} : {len(df) - len(valid_values)} invalid values")


### Removing useless columns and Renaming columns


In [56]:
launches_df = launches_df[[
    'id', 'name', 'date_utc', 'flight_number',
    'success', 'upcoming', 'rocket', 'launchpad', 'details'
]]


rockets_df = rockets_df[[
    'id', 'name', 'type', 'active',
    'stages', 'boosters', 'cost_per_launch',
    'success_rate_pct', 'country', 'company',
    'first_flight'
]]

launchpads_df = launchpads_df[[
    'id', 'name', 'locality', 'region',
    'timezone', 'launch_attempts', 'launch_successes'
]]

launches_df = launches_df.rename(columns={
    'id': 'launch_id',
    'name': 'mission_name',
    'rocket': 'rocket_id',
    'launchpad': 'launchpad_id'
})

rockets_df = rockets_df.rename(columns={
    'id': 'rocket_id',
    'name': 'rocket_name',
    'active': 'active_flag'
})

launchpads_df = launchpads_df.rename(columns={
    'id': 'launchpad_id',
    'name': 'launchpad_name'
})

### Creating New columns

In [57]:
launches_df['date_utc'] = pd.to_datetime(launches_df['date_utc'])
launches_df['launch_date'] = launches_df['date_utc'].dt.date
launches_df['launch_year'] = launches_df['date_utc'].dt.year
launches_df['launch_month'] = launches_df['date_utc'].dt.month

launches_df['launch_status'] = launches_df.apply(get_launch_status, axis=1)

### checking nulls and duplicate , if there are duplicates removing them  

In [58]:
check_duplicates(launches_df , 'launch_id')
check_all_nulls(launches_df)


Duplicates in launch_id : 0
nulls : launch_id         0
mission_name      0
date_utc          0
flight_number     0
success          19
upcoming          0
rocket_id         0
launchpad_id      0
details          71
launch_date       0
launch_year       0
launch_month      0
launch_status     0
dtype: int64


In [59]:
check_duplicates(rockets_df,'rocket_id')
check_all_nulls(rockets_df)

Duplicates in rocket_id : 0
nulls : rocket_id           0
rocket_name         0
type                0
active_flag         0
stages              0
boosters            0
cost_per_launch     0
success_rate_pct    0
country             0
company             0
first_flight        0
dtype: int64


In [60]:
check_duplicates(launchpads_df , 'launchpad_id')
check_all_nulls(launchpads_df)

Duplicates in launchpad_id : 0
nulls : launchpad_id        0
launchpad_name      0
locality            0
region              0
timezone            0
launch_attempts     0
launch_successes    0
dtype: int64


### Range Checks , key identifiers are not null , referential check and success col check

In [61]:
range_check(rockets_df , 'success_rate_pct' , 0 , 100)

range_check(rockets_df , 'cost_per_launch' , 0 , max(rockets_df['cost_per_launch']))

range_check(rockets_df , 'stages', 0 , max(rockets_df['stages']))

range_check(rockets_df , 'boosters', 0 , max(rockets_df['boosters']))

range_check(launches_df , 'launch_year', 1980 , 2026)

range_check(launches_df , 'launch_month', 1 , 12)

range_check(launches_df , 'flight_number', 1 , max(launches_df['flight_number']))

range_check(launchpads_df , 'launch_attempts', 0 , max(launchpads_df['launch_attempts']))

range_check(launchpads_df , 'launch_successes', 0 , max(launchpads_df['launch_successes']))

referenence_check(launches_df , rockets_df , 'rocket_id' , 'rocket_id')


referenence_check(launches_df , launchpads_df , 'launchpad_id' , 'launchpad_id')

success_col_check(launches_df , 'success')

Range check for success_rate_pct in range [0, 100]: 0 out of range values
Range check for cost_per_launch in range [0, 90000000]: 0 out of range values
Range check for stages in range [0, 2]: 0 out of range values
Range check for boosters in range [0, 2]: 0 out of range values
Range check for launch_year in range [1980, 2026]: 0 out of range values
Range check for launch_month in range [1, 12]: 0 out of range values
Range check for flight_number in range [1, 203]: 0 out of range values
Range check for launch_attempts in range [0, 99]: 0 out of range values
Range check for launch_successes in range [0, 97]: 0 out of range values
Reference check for rocket_id in rocket_id : 0 unmatched values
Reference check for launchpad_id in launchpad_id : 0 unmatched values
Success column check for success : 0 invalid values


### Saving as parquet 

In [62]:
def save_parquet(df , name):
    df.to_parquet(f'{name}.parquet')

In [63]:
save_parquet(launches_df , 'launches')
save_parquet(launchpads_df , 'launchpads')
save_parquet(rockets_df , 'rockets')